# Distillation Basics

This notebook teaches a small student model using a larger teacher.

Goals:
- train a teacher on the full dataset
- train two students on a small subset
- compare hard-label training vs distillation


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

digits = load_digits()
X = torch.tensor(digits.data / 16.0, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_small = X_train_full[:200]
y_train_small = y_train_full[:200]

full_loader = DataLoader(TensorDataset(X_train_full, y_train_full), batch_size=64, shuffle=True)
small_loader = DataLoader(TensorDataset(X_train_small, y_train_small), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)


In [ ]:
class Teacher(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


class Student(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        return self.net(x)


def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
    return correct / total


teacher = Teacher()
student_hard = Student()
student_distilled = Student()


In [ ]:
teacher_optimizer = torch.optim.Adam(teacher.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(10):
    teacher.train()
    for xb, yb in full_loader:
        loss = loss_fn(teacher(xb), yb)
        teacher_optimizer.zero_grad()
        loss.backward()
        teacher_optimizer.step()

teacher_acc = evaluate(teacher)
teacher_acc


In [ ]:
hard_optimizer = torch.optim.Adam(student_hard.parameters(), lr=0.01)

for epoch in range(15):
    student_hard.train()
    for xb, yb in small_loader:
        loss = loss_fn(student_hard(xb), yb)
        hard_optimizer.zero_grad()
        loss.backward()
        hard_optimizer.step()

student_hard_acc = evaluate(student_hard)
student_hard_acc


In [ ]:
distill_optimizer = torch.optim.Adam(student_distilled.parameters(), lr=0.01)
temperature = 3.0
alpha = 0.5

teacher.eval()
for epoch in range(15):
    student_distilled.train()
    for xb, yb in small_loader:
        with torch.no_grad():
            teacher_logits = teacher(xb)

        student_logits = student_distilled(xb)
        soft_loss = F.kl_div(
            F.log_softmax(student_logits / temperature, dim=1),
            F.softmax(teacher_logits / temperature, dim=1),
            reduction="batchmean",
        ) * (temperature ** 2)
        hard_loss = loss_fn(student_logits, yb)
        loss = alpha * soft_loss + (1 - alpha) * hard_loss

        distill_optimizer.zero_grad()
        loss.backward()
        distill_optimizer.step()

student_distilled_acc = evaluate(student_distilled)

def count_params(model):
    return sum(parameter.numel() for parameter in model.parameters())


pd.DataFrame(
    [
        {"model": "teacher", "params": count_params(teacher), "accuracy": teacher_acc},
        {"model": "student_hard", "params": count_params(student_hard), "accuracy": student_hard_acc},
        {"model": "student_distilled", "params": count_params(student_distilled), "accuracy": student_distilled_acc},
    ]
)
